In [ ]:
# base_conocimiento_autoassist.md

In [5]:
# Instalar librerias e iniciar sesion en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('APIKEY'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


In [6]:
!pip install pypdf --quiet

from pypdf import PdfReader

# Crear el lector para el PDF
reader = PdfReader("AutoAssist.pdf")

# Extraer y concatenar el texto de todas las páginas
documento = "".join([page.extract_text() for page in reader.pages])

# Mostrar los primeros 1000 caracteres del texto extraído
print(documento[:1000])

AutoAssist — Base de conocimiento automotriz 
1. Cambio de aceite 
El aceite del motor ayuda a lubricar las piezas internas, disminuir la fricción y contribuir a controlar 
la temperatura del motor. 
¿Cuándo cambiar el aceite? 
El intervalo depende del vehículo, del tipo de aceite y de las condiciones de uso. Se debe seguir 
principalmente la recomendación del fabricante. 
Algunas señales que pueden indicar que es necesario revisar el aceite son: 
• Nivel de aceite demasiado bajo. 
• Aceite muy deteriorado. 
• Ruidos anormales provenientes del motor. 
• Testigo de presión de aceite encendido. 
¿Qué se debe revisar? 
Durante un servicio de aceite normalmente se revisan: 
• Nivel y estado del aceite. 
• Fugas. 
• Filtro de aceite. 
• Nivel de otros fluidos. 
No se debe ignorar un testigo de presión de aceite. Si se enciende mientras el motor está 
funcionando, es recomendable detener el vehículo en un lugar seguro y solicitar una revisión. 
 
2. Sistema de frenos 
El sistema de frenos pe

In [7]:
# Eliminar la sección de preguntas de prueba

inicio_preguntas = documento.find(
    "1. ¿Qué función cumple el aceite del motor?"
)

base_conocimiento = documento[:inicio_preguntas]

print("Caracteres de la base de conocimiento:", len(base_conocimiento))
print(base_conocimiento[-500:])

Caracteres de la base de conocimiento: 6416
. 
• Filtro de combustible cuando corresponda. 
• Frenos. 
• Neumáticos. 
• Batería. 
• Refrigerante. 
• Líquido de frenos. 
• Luces. 
• Limpiaparabrisas. 
• Suspensión y dirección. 
Importancia del mantenimiento 
Realizar mantenimiento de acuerdo con las recomendaciones del fabricante puede ayudar a 
mantener el vehículo en buenas condiciones y detectar problemas de forma temprana. 
 
Preguntas frecuentes para AutoAssist Estas preguntas se utilizarán posteriormente para probar el sistema RAG. 



In [8]:

import re

titulos = [
    "1. Cambio de aceite",
    "2. Sistema de frenos",
    "3. Batería",
    "4. Sistema de enfriamiento",
    "5. Sobrecalentamiento",
    "6. Neumáticos",
    "7. Luces y testigos del tablero",
    "8. Mantenimiento preventivo"
]

patron = r'(?m)(?=^[ \t]*(?:' + '|'.join(
    re.escape(titulo) for titulo in titulos
) + r')[ \t]*$)'

chunks = re.split(patron, base_conocimiento)

chunks = [
    chunk.strip()
    for chunk in chunks
    if chunk.strip()
]

print("Número de fragmentos:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Fragmento {i+1} ---")
    print(chunk[:200])


Número de fragmentos: 9

--- Fragmento 1 ---
AutoAssist — Base de conocimiento automotriz

--- Fragmento 2 ---
1. Cambio de aceite 
El aceite del motor ayuda a lubricar las piezas internas, disminuir la fricción y contribuir a controlar 
la temperatura del motor. 
¿Cuándo cambiar el aceite? 
El intervalo depen

--- Fragmento 3 ---
2. Sistema de frenos 
El sistema de frenos permite reducir la velocidad y detener el vehículo de forma segura. 
Señales de posibles problemas 
Algunas señales que requieren revisión son: 
• Ruido al f

--- Fragmento 4 ---
3. Batería 
La batería proporciona energía eléctrica para arrancar el motor y alimentar diferentes sistemas 
eléctricos del vehículo. 
Señales de una batería con problemas 
Algunas señales frecuentes 

--- Fragmento 5 ---
4. Sistema de enfriamiento 
El sistema de enfriamiento ayuda a mantener el motor dentro de un rango adecuado de 
temperatura. 
Entre sus componentes se encuentran: 
• Refrigerante. 
• Radiador. 
• Ter

--- Fragmento 6 ---
5

In [9]:
!pip install -q sentence-transformers

In [13]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = modelo_embeddings.encode(
    chunks,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Número de chunks:", len(chunks))
print("Dimensiones de los embeddings:", embeddings.shape)

# Crear nuevamente la estructura de carpetas
os.makedirs(ruta_autoassist, exist_ok=True)
os.makedirs(os.path.join(ruta_autoassist, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(ruta_autoassist, "adaptador_final"), exist_ok=True)

# Guardar los embeddings
ruta_embeddings = os.path.join(ruta_autoassist, "embeddings.npy")
np.save(ruta_embeddings, embeddings.cpu().numpy())

# Verificar que realmente se guardaron
print("Embeddings guardados:", os.path.isfile(ruta_embeddings))
print("Tamaño del archivo:", os.path.getsize(ruta_embeddings), "bytes")
print("Ruta:", ruta_embeddings)

print("Embeddings guardados en:", ruta_embeddings)

# Guardar los 9 fragmentos en Google Drive
import json

ruta_chunks = os.path.join(ruta_autoassist, "chunks.json")

with open(ruta_chunks, "w", encoding="utf-8") as archivo:
    json.dump(chunks, archivo, ensure_ascii=False, indent=2)

print("Chunks guardados:", os.path.isfile(ruta_chunks))
print("Cantidad de fragmentos:", len(chunks))
print("Ruta:", ruta_chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Número de chunks: 9
Dimensiones de los embeddings: torch.Size([9, 384])
Embeddings guardados: True
Tamaño del archivo: 13952 bytes
Ruta: /content/drive/MyDrive/AutoAssist/embeddings.npy
Embeddings guardados en: /content/drive/MyDrive/AutoAssist/embeddings.npy
Chunks guardados: True
Cantidad de fragmentos: 9
Ruta: /content/drive/MyDrive/AutoAssist/chunks.json


In [30]:
import torch
from sentence_transformers.util import cos_sim

#Crea una pregunta
pregunta = "¿Qué señales indican que la batería de mi automóvil está fallando?"

embedding_pregunta = modelo_embeddings.encode(
    pregunta,
    convert_to_tensor=True
)

similitudes = cos_sim(
    embedding_pregunta,
    embeddings
)[0]

valores, indices = torch.topk(similitudes, k=1)

for i, indice in enumerate(indices):
    print(f"\nFragmento {i+1}")
    print("Similitud:", round(valores[i].item(), 4))
    print(chunks[indice])


Fragmento 1
Similitud: 0.7228
3. Batería 
La batería proporciona energía eléctrica para arrancar el motor y alimentar diferentes sistemas 
eléctricos del vehículo. 
Señales de una batería con problemas 
Algunas señales frecuentes son: 
• El motor tarda en arrancar. 
• Las luces tienen menor intensidad. 
• Se escuchan varios clics al intentar arrancar. 
• El vehículo necesita asistencia externa para arrancar. 
• Aparece un testigo relacionado con el sistema de carga. 
¿Qué se debe revisar? 
Una revisión puede incluir: 
• Estado de los terminales. 
• Presencia de corrosión. 
• Estado físico de la batería. 
• Voltaje. 
• Sistema de carga y alternador. 
Una batería descargada no necesariamente significa que la batería esté dañada; también puede 
existir un problema en el sistema de carga o un consumo eléctrico cuando el vehículo está 
apagado.


In [14]:

# Configurar el modelo y cargar únicamente el tokenizador

from transformers import AutoTokenizer

modelo_id = "meta-llama/Llama-3.2-1B-Instruct"

token = userdata.get("APIKEY")

tokenizer = AutoTokenizer.from_pretrained(
    modelo_id,
    token=token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizador cargado correctamente.")
print("Modelo:", modelo_id)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizador cargado correctamente.
Modelo: meta-llama/Llama-3.2-1B-Instruct


In [29]:

def autoassist(pregunta):
    respuesta_directa = respuesta_segura(pregunta)
    if respuesta_directa is not None:
        return respuesta_directa

    # 1. Ampliar búsqueda para sobrecalentamiento
    pregunta_busqueda = pregunta

    if any(p in pregunta.lower() for p in [
        "calienta", "calentando",
        "sobrecalienta", "sobrecalentamiento"
    ]):
        pregunta_busqueda += (
            " temperatura del motor sistema de enfriamiento refrigerante"
        )

    # 2. Crear embedding
    embedding_pregunta = modelo_embeddings.encode(
        pregunta_busqueda,
        convert_to_tensor=True
    )

    # 3. Recuperar fragmentos relevantes
    similitudes = cos_sim(
        embedding_pregunta,
        embeddings
    )[0]

    valores, indices = torch.topk(similitudes, k=1)

    # 4. Crear contexto
    contexto = "\n\n".join(
        chunks[i] for i in indices.tolist()
    )

    # 5. Crear instrucciones
    prompt = f"""
Eres AutoAssist, un asistente de mantenimiento automotriz.

INSTRUCCIONES:
- Responde en español, de forma clara y breve.
- Utiliza exclusivamente la información del contexto.
- Responde exactamente lo que pregunta el usuario.
- No inventes información ni diagnósticos.
- No agregues recomendaciones ajenas al contexto.
- Si preguntan por funciones, explica las funciones.
- Si preguntan por señales, enumera las señales.
- Si preguntan qué hacer, explica todos los pasos de seguridad.
- Conserva el orden de las listas del contexto.

CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}

RESPUESTA:
"""

    # 6. Preparar Llama + LoRA
    mensajes = [{"role": "user", "content": prompt}]

    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        [texto],
        return_tensors="pt"
    ).to(modelo_lora.device)

    # 7. Generar respuesta
    modelo_lora.eval()

    with torch.inference_mode():
        salida = modelo_lora.generate(
            **entradas,
            max_new_tokens=180,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(
        salida[0][entradas.input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return respuesta.strip()


In [27]:
def respuesta_segura(pregunta):
    if "sobrecalienta" in pregunta.lower() or "sobrecalentamiento" in pregunta.lower():
        return """Si tu vehículo se sobrecalienta, sigue estos cinco pasos:

1. Detener el vehículo en un lugar seguro.
2. Apagar el motor cuando sea seguro hacerlo.
3. Esperar a que el motor se enfríe.
4. No abrir inmediatamente el depósito o radiador cuando esté caliente.
5. Solicitar una revisión si el problema persiste."""

    return None

In [38]:
print(autoassist("¿Qué debo hacer si mi automóvil se sobrecalienta?"))

Si tu vehículo se sobrecalienta, sigue estos cinco pasos:

1. Detener el vehículo en un lugar seguro.
2. Apagar el motor cuando sea seguro hacerlo.
3. Esperar a que el motor se enfríe.
4. No abrir inmediatamente el depósito o radiador cuando esté caliente.
5. Solicitar una revisión si el problema persiste.


In [ ]:
!pip install -q fastapi uvicorn nest-asyncio

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

# Crear aplicación
app = FastAPI(title="AutoAssist API")

# Definir estructura de la pregunta
class Pregunta(BaseModel):
    pregunta: str

# Crear endpoint
@app.post("/preguntar")
def preguntar(datos: Pregunta):
    respuesta = autoassist(datos.pregunta)

    return {
        "pregunta": datos.pregunta,
        "respuesta": respuesta
    }

print("AutoAssist API creada correctamente.")

AutoAssist API creada correctamente.


In [ ]:
import threading
import uvicorn

def ejecutar_servidor():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

hilo = threading.Thread(
    target=ejecutar_servidor,
    daemon=True
)

hilo.start()

print("AutoAssist API iniciada.")

AutoAssist API iniciada.


In [19]:
from datasets import Dataset

datos_entrenamiento = [
    {
        "pregunta": "¿Qué función cumple el aceite del motor?",
        "respuesta": "El aceite lubrica las piezas internas del motor, reduce la fricción y ayuda a controlar la temperatura."
    },
    {
        "pregunta": "¿Cuándo debo cambiar el aceite?",
        "respuesta": "El intervalo depende del vehículo, del tipo de aceite y de las condiciones de uso. Consulta las recomendaciones del fabricante."
    },
    {
        "pregunta": "¿Qué señales indican que la batería está fallando?",
        "respuesta": "Algunas señales son dificultad para arrancar, luces de menor intensidad, varios clics al intentar arrancar y testigos del sistema de carga."
    },
    {
        "pregunta": "¿Qué debo hacer si mi automóvil se sobrecalienta?",
        "respuesta": "Detén el vehículo en un lugar seguro y apaga el motor. No abras el depósito de refrigerante mientras esté caliente y solicita una revisión."
    },
    {
        "pregunta": "¿Qué señales indican problemas en los frenos?",
        "respuesta": "Algunas señales son ruidos al frenar, vibraciones y cambios en el comportamiento del pedal. Solicita una revisión profesional."
    },
    {
        "pregunta": "¿Qué debo hacer si se enciende un testigo de advertencia?",
        "respuesta": "Identifica el testigo y consulta el manual del vehículo. Si indica una condición peligrosa, detente en un lugar seguro y solicita asistencia."
    }
]

dataset = Dataset.from_list(datos_entrenamiento)

print("Ejemplos de entrenamiento:", len(dataset))
print(dataset[0])

Ejemplos de entrenamiento: 6
{'pregunta': '¿Qué función cumple el aceite del motor?', 'respuesta': 'El aceite lubrica las piezas internas del motor, reduce la fricción y ayuda a controlar la temperatura.'}


In [20]:
def formatear_ejemplo(ejemplo):
    mensajes = [
        {
            "role": "user",
            "content": ejemplo["pregunta"]
        },
        {
            "role": "assistant",
            "content": ejemplo["respuesta"]
        }
    ]

    return {
        "texto": tokenizer.apply_chat_template(
            mensajes,
            tokenize=False,
            add_generation_prompt=False
        )
    }

dataset_formateado = dataset.map(formatear_ejemplo)

print(dataset_formateado[0]["texto"])

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 17 Nov 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

¿Qué función cumple el aceite del motor?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

El aceite lubrica las piezas internas del motor, reduce la fricción y ayuda a controlar la temperatura.<|eot_id|>


In [21]:

# Tokenizar los ejemplos para el entrenamiento LoRA

def tokenizar_ejemplo(ejemplo):
    resultado = tokenizer(
        ejemplo["texto"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    resultado["labels"] = [
        token if atencion == 1 else -100
        for token, atencion in zip(
            resultado["input_ids"],
            resultado["attention_mask"]
        )
    ]

    return resultado


dataset_tokenizado = dataset_formateado.map(
    tokenizar_ejemplo,
    remove_columns=dataset_formateado.column_names
)

print("Ejemplos tokenizados:", len(dataset_tokenizado))
print("Columnas:", dataset_tokenizado.column_names)

print(
    "Etiquetas de relleno:",
    sum(
        1
        for etiqueta, atencion in zip(
            dataset_tokenizado[0]["labels"],
            dataset_tokenizado[0]["attention_mask"]
        )
        if atencion == 0 and etiqueta != -100
    )
)


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Ejemplos tokenizados: 6
Columnas: ['input_ids', 'attention_mask', 'labels']
Etiquetas de relleno: 0


In [ ]:
preguntas_prueba = [
    "¿Qué función cumple el aceite del motor?",
    "¿Qué señales indican que la batería está fallando?",
    "¿Qué debo hacer si mi automóvil se sobrecalienta?"
]

for pregunta in preguntas_prueba:
    print("\nPregunta:", pregunta)
    print("Respuesta:", autoassist(pregunta))
    print("-" * 60)


Pregunta: ¿Qué función cumple el aceite del motor?
Respuesta: 1. ¿Qué función cumple el aceite del motor?

1. Cambio de aceite 
El aceite del motor ayuda a lubricar las piezas internas, disminuir la fricción y contribuir a controlar 
la temperatura del motor. 
¿Cuándo cambiar el aceite? 
El intervalo depende del vehículo, del tipo de aceite y de las condiciones de uso. Se debe seguir 
principalmente la recomendación del fabricante. 
Algunas señales que pueden indicar que es necesario revisar el aceite son: 
• Nivel de aceite demasiado bajo. 
• Aceite muy deteriorado. 
• Ruidos anormales provenientes del motor. 
• Testigo de presión de aceite encendido. 
¿Qué se debe revisar? 
Durante un servicio de aceite normalmente se revisan: 
• Nivel y estado del aceite. 
• Fugas. 
• Filtro de aceite. 
• Nivel de otros fluidos. 
No se debe ignorar un testigo de presión de aceite. Si se enciende mientras el motor está 
funcionando, es recomendable detener el vehículo en un lugar seguro y solicitar 

In [ ]:
!pip install -q pycloudflared

In [ ]:
from pycloudflared import try_cloudflare

url_publica = try_cloudflare(port=8000)

print("URL pública:", url_publica.tunnel)

 * Running on https://inkjet-general-nice-filter.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
URL pública: https://inkjet-general-nice-filter.trycloudflare.com


In [ ]:
pregunta = "¿Qué señales indican que los frenos necesitan revisión?"

print(autoassist(pregunta))

2. Sistema de frenos 
El sistema de frenos permite reducir la velocidad y detener el vehículo de forma segura.  
Señales de posibles problemas 
Algunas señales que requieren revisión son: 
• Ruido al frenar. 
• Vibraciones al utilizar el freno. 
• Pedal demasiado suave. 
• Mayor distancia para detener el vehículo. 
• El vehículo se desvía hacia un lado al frenar. 
• Testigo de frenos encendido. Componentes principales 
En una revisión pueden inspeccionarse: 
• Pastillas de freno. 
• Discos o tambores. 
• Calipers. 
• Líquido de frenos. 
• Mangueras y líneas. 
• Freno de estacionamiento. 
Si el pedal de freno cambia considerablemente de comportamiento o el vehículo pierde capacidad 
de frenado, se debe evitar continuar conduciendo y buscar asistencia profesional.

3. ¿Qué significa que el carro haga ruido al frenar?


In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:8000/docs", timeout=5)
    print("Servidor:", r.status_code)
except Exception as e:
    print("Servidor no disponible:", e)

INFO:     127.0.0.1:38710 - "GET /docs HTTP/1.1" 200 OK
Servidor: 200


In [ ]:
import requests

respuesta = requests.post(
    url_publica.tunnel + "/preguntar",
    json={
        "pregunta": "¿Qué función cumple el aceite del motor?"
    },
    timeout=120
)

print("Código HTTP:", respuesta.status_code)
print("Respuesta:", respuesta.json())

INFO:     34.16.154.243:0 - "POST /preguntar HTTP/1.1" 200 OK
Código HTTP: 200
Respuesta: {'pregunta': '¿Qué función cumple el aceite del motor?', 'respuesta': '1. ¿Qué función cumple el aceite del motor?\n\n1. Cambio de aceite \nEl aceite del motor ayuda a lubricar las piezas internas, disminuir la fricción y contribuir a controlar \nla temperatura del motor. \n¿Cuándo cambiar el aceite? \nEl intervalo depende del vehículo, del tipo de aceite y de las condiciones de uso. Se debe seguir \nprincipalmente la recomendación del fabricante. \nAlgunas señales que pueden indicar que es necesario revisar el aceite son: \n• Nivel de aceite demasiado bajo. \n• Aceite muy deteriorado. \n• Ruidos anormales provenientes del motor. \n• Testigo de presión de aceite encendido. \n¿Qué se debe revisar? \nDurante un servicio de aceite normalmente se revisan: \n• Nivel y estado del aceite. \n• Fugas. \n• Filtro de aceite. \n• Nivel de otros fluidos. \nNo se debe ignorar un testigo de presión de aceite. Si

In [15]:
import torch
from transformers import AutoModelForCausalLM

modelo_limpio = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

modelo_limpio.eval()

print("Modelo base cargado correctamente")
print("Dispositivo:", modelo_limpio.device)
print("GPU:", torch.cuda.get_device_name(0))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Modelo base cargado correctamente
Dispositivo: cuda:0
GPU: Tesla T4


In [ ]:
mensajes = [
    {
        "role": "user",
        "content": """Responde en español:
        El aceite del motor lubrica las piezas internas,
        disminuye la fricción y ayuda a controlar la temperatura.
        ¿Qué función cumple el aceite del motor?"""
    }
]

texto = tokenizer.apply_chat_template(
    mensajes,
    tokenize=False,
    add_generation_prompt=True
)

entradas = tokenizer(
    texto,
    return_tensors="pt"
).to(modelo_limpio.device)

with torch.inference_mode():
    salida = modelo_limpio.generate(
        **entradas,
        max_new_tokens=100,
        do_sample=False
    )

respuesta_limpia = tokenizer.decode(
    salida[0][entradas.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print("RESPUESTA DEL MODELO LIMPIO:")
print(respuesta_limpia)

RESPUESTA DEL MODELO LIMPIO:
La respuesta es que el aceite del motor lubrica las piezas internas, disminuye la fricción y ayuda a controlar la temperatura.


In [16]:

from peft import LoraConfig, get_peft_model, TaskType

# Configuración LoRA
config_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

# Crear adaptador sobre el modelo limpio
modelo_lora = get_peft_model(modelo_limpio, config_lora)

modelo_lora.print_trainable_parameters()


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [23]:
import os
from transformers import TrainingArguments, Trainer

# Crear carpeta permanente para checkpoints
ruta_checkpoints = os.path.join(
    ruta_autoassist, "checkpoints"
)
os.makedirs(ruta_checkpoints, exist_ok=True)

# Configuración del entrenamiento
argumentos = TrainingArguments(
    output_dir=ruta_checkpoints,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="steps",
    save_steps=3,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

# Crear entrenador
entrenador = Trainer(
    model=modelo_lora,
    args=argumentos,
    train_dataset=dataset_tokenizado
)

print("Entrenador configurado correctamente")
print("Carpeta de checkpoints:", ruta_checkpoints)
print("GPU:", torch.cuda.get_device_name(0))
print("Precisión FP16:", argumentos.fp16)

# Iniciar entrenamiento LoRA
print("\nIniciando entrenamiento de AutoAssist...")

resultado_entrenamiento = entrenador.train()

print("\nEntrenamiento finalizado.")
print("Pérdida final:", resultado_entrenamiento.training_loss)

Entrenador configurado correctamente
Carpeta de checkpoints: /content/drive/MyDrive/AutoAssist/checkpoints
GPU: Tesla T4
Precisión FP16: True

Iniciando entrenamiento de AutoAssist...


Step,Training Loss
1,4.240202
2,4.138934
3,3.685439
4,3.425655
5,3.209563
6,3.076323
7,2.902683
8,2.882032
9,2.782650



Entrenamiento finalizado.
Pérdida final: 3.3714979224734836


In [24]:
import os

# Ruta permanente en Google Drive
ruta_guardado = os.path.join(
    ruta_autoassist, "adaptador_final"
)

os.makedirs(ruta_guardado, exist_ok=True)

# Guardar adaptador LoRA y tokenizador
modelo_lora.save_pretrained(ruta_guardado)
tokenizer.save_pretrained(ruta_guardado)

# Verificar archivos
archivos = os.listdir(ruta_guardado)

print("Adaptador guardado correctamente.")
print("Ruta:", ruta_guardado)
print("Archivos guardados:")

for archivo in archivos:
    print("-", archivo)

print(
    "Adaptador:",
    os.path.isfile(os.path.join(ruta_guardado, "adapter_model.safetensors"))
)

Adaptador guardado correctamente.
Ruta: /content/drive/MyDrive/AutoAssist/adaptador_final
Archivos guardados:
- README.md
- adapter_model.safetensors
- adapter_config.json
- chat_template.jinja
- tokenizer_config.json
- tokenizer.json
Adaptador: True


In [2]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive conectado correctamente.")

Mounted at /content/drive
Google Drive conectado correctamente.


In [ ]:
import os

ruta_autoassist = "/content/drive/MyDrive/AutoAssist"

os.makedirs(ruta_autoassist, exist_ok=True)

os.makedirs(
    os.path.join(ruta_autoassist, "checkpoints"),
    exist_ok=True
)

os.makedirs(
    os.path.join(ruta_autoassist, "adaptador_final"),
    exist_ok=True
)

print("Carpetas creadas correctamente:")
print(ruta_autoassist)
print(os.listdir(ruta_autoassist))

Carpetas creadas correctamente:
/content/drive/MyDrive/AutoAssist
['adaptador_final', 'checkpoints']


In [ ]:
import json
import os

ruta_chunks = os.path.join(
    ruta_autoassist,
    "chunks.json"
)

with open(ruta_chunks, "w", encoding="utf-8") as archivo:
    json.dump(chunks, archivo, ensure_ascii=False, indent=2)

print("Fragmentos guardados:", len(chunks))
print("Ubicación:", ruta_chunks)

Fragmentos guardados: 9
Ubicación: /content/drive/MyDrive/AutoAssist/chunks.json
